# 📊 Case Item 07 — Análise de Dados, Dashboards (Metabase) & Camada Semântica
### Data Lakehouse Dadosfera & Serving Analítico

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pedrosales/PEDRO_SALES_DDF_TECH_082026/blob/main/pipelines/case-item-07/notebooks/07_bi_dashboards_visualizations.ipynb)

> **Módulo:** `pipelines/case-item-07/`  
> **Papel Arquitetural:** Hub Central de Análise de Dados, Visualizações de BI e Camada Semântica  
> **Frameworks & Padrões:** Padrão Visual `charts-maker` (Fundo Branco `#FFFFFF`, 300 DPI) • DEC-001 (Métricas em %) • DEC-008 (Kimball DW) • Metabase Snowflake Queries  
> **Status:** Executável ponta a ponta com persistência e leitura de Ground Truth em Parquet.

---

## 📋 Sumário Executivo
Este notebook consolida a camada de **Serving e Análise Visual de BI** para o case de **Recuperação de Carrinho Abandonado**, cumprindo integralmente os requisitos do **Item 7 do Case Técnico Dadosfera**:
1. **Análise de Série Temporal:** Evolução diária/semanal de taxas de abandono vs recuperação e GMV.
2. **Análise de Categorias de Produto:** Identificação de volume e gargalos de conversão no catálogo.
3. **Rentabilidade & ROI por Canal:** Cruzamento de CAC de resgate e faturamento gerado.
4. **Matriz de Atrito RFM:** Heatmap de causas-raiz de abandono cruzadas com segmentos de clientes.
5. **Matriz Prescritiva de Viabilidade:** Scatter plot para priorização em réguas de CRM e Data Apps.
6. **Scorecard de Data Quality:** Transparência de governança entre dados conformes (Silver) e Quarentena.

In [ ]:
# 📦 1. Configuração do Ambiente e Importações
import os
import sys
from typing import Dict, Any, Tuple
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Configuração de Estilo charts-maker Standard (Fundo Branco #FFFFFF)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Helvetica", "Arial", "sans-serif"],
    "axes.edgecolor": "#CBD5E1",
    "axes.linewidth": 1.0,
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.45,
})

print("✅ Ambiente configurado com sucesso para execução de BI e Visualizações!")

In [ ]:
# 📥 2. Carga dos Datasets Limpos do Lakehouse (Ground Truth)
import os

def find_base_dir() -> str:
    for candidate in [".", "..", "../..", "../../.."]:
        if os.path.exists(os.path.join(candidate, "data", "mock", "output_cleaned", "parquet")):
            return os.path.abspath(candidate)
    return os.path.abspath(".")

BASE_DIR = find_base_dir()
PARQUET_DIR = os.path.join(BASE_DIR, "data", "mock", "output_cleaned", "parquet")
FALLBACK_DIR = os.path.join(BASE_DIR, "data", "mock", "output", "parquet")

def load_entity(name: str) -> pd.DataFrame:
    path = os.path.join(PARQUET_DIR, f"{name}.parquet")
    if not os.path.exists(path):
        path = os.path.join(FALLBACK_DIR, f"{name}.parquet")
    if os.path.exists(path):
        return pd.read_parquet(path)
    return pd.DataFrame()

df_carrinhos = load_entity("carrinhos")
df_pedidos = load_entity("pedidos")
df_clientes = load_entity("clientes")
df_resgate = load_entity("eventos_resgate")
df_produtos = load_entity("produtos")
df_itens = load_entity("itens_carrinho")

print(f"✅ Datasets carregados com sucesso:")
print(f"  • Carrinhos: {len(df_carrinhos):,} registros")
print(f"  • Itens de Carrinho: {len(df_itens):,} registros")
print(f"  • Produtos no Catálogo: {len(df_produtos):,} registros")
print(f"  • Disparos de Resgate: {len(df_resgate):,} registros")
print(f"  • Clientes: {len(df_clientes):,} registros")

---  
### 📊 Visualização 1: Série Temporal — Evolução Semanal de Abandono vs Recuperação
Atendimento ao requisito explícito de **Série Temporal** do Item 7.

In [ ]:
# 📈 3. Query Analítica & Plot: Série Temporal
fig, ax1 = plt.subplots(figsize=(11, 5.5), facecolor="#FFFFFF")
ax1.set_facecolor("#FFFFFF")

df = df_carrinhos.copy()
df["data_criacao"] = pd.to_datetime(df["data_criacao"])
if df["data_criacao"].dt.tz is not None:
    df["data_criacao"] = df["data_criacao"].dt.tz_localize(None)
df["semana"] = df["data_criacao"].dt.to_period("W").apply(lambda r: r.start_time)

agg = df.groupby("semana").agg(
    total=("carrinho_id", "count"),
    abandonados=("status", lambda s: (s == "abandonado").sum()),
    recuperados=("status", lambda s: (s == "recuperado").sum()),
    gmv_total=("valor_total", "sum"),
    gmv_recuperado=("valor_total", lambda v: v[df.loc[v.index, "status"] == "recuperado"].sum())
).reset_index()

agg["taxa_abandono"] = (agg["abandonados"] / agg["total"]) * 100.0
agg["taxa_recuperacao"] = (agg["recuperados"] / agg["abandonados"].replace(0, 1)) * 100.0

x_labels = [d.strftime("%d/%b") for d in agg["semana"]]
x = np.arange(len(agg))

l1, = ax1.plot(x, agg["taxa_abandono"], color="#E11D48", linewidth=2.4, marker="o", markersize=5, label="Taxa de Abandono (%)")
l2, = ax1.plot(x, agg["taxa_recuperacao"], color="#059669", linewidth=2.4, marker="s", markersize=5, label="Taxa de Recuperação (%)")
ax1.fill_between(x, agg["taxa_abandono"], alpha=0.10, color="#E11D48")
ax1.fill_between(x, agg["taxa_recuperacao"], alpha=0.15, color="#059669")

ax1.set_ylabel("Taxa Percentual (%)", fontsize=11, fontweight="bold", color="#1E293B")
ax1.set_xticks(x[::2])
ax1.set_xticklabels(x_labels[::2], fontsize=9.5, fontweight="bold", color="#334155")
ax1.set_ylim(0, 100)
ax1.grid(True, linestyle="--", alpha=0.45)
ax1.spines["top"].set_visible(False)

ax2 = ax1.twinx()
ax2.set_facecolor("#FFFFFF")
l3, = ax2.plot(x, agg["gmv_recuperado"] / 1000.0, color="#2563EB", linewidth=1.8, linestyle="--", label="GMV Recuperado (R$ mil)")
ax2.set_ylabel("GMV Recuperado (R$ mil)", fontsize=11, fontweight="bold", color="#2563EB")
ax2.spines["top"].set_visible(False)

ax1.set_title("Evolução Semanal de Abandono vs Recuperação de Carrinhos (2026)", fontsize=13, fontweight="bold", color="#0F172A", pad=12)
lines = [l1, l2, l3]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper right", frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1", fontsize=9)
plt.show()

---  
### 📊 Visualização 2: Performance de Catálogo por Categoria de Produto
Atendimento ao requisito explícito de **Análise por Categorias** do Item 7.

In [ ]:
# 🏷️ 4. Query Analítica & Plot: Performance de Categorias
fig, ax = plt.subplots(figsize=(10.5, 5.5), facecolor="#FFFFFF")
ax.set_facecolor("#FFFFFF")

preco_col = "preco_atual" if "preco_atual" in df_produtos.columns else ("preco" if "preco" in df_produtos.columns else None)
if not df_itens.empty and not df_produtos.empty and not df_carrinhos.empty and preco_col:
    cols_prod = ["produto_id", "categoria", preco_col]
    merged = df_itens.merge(df_produtos[cols_prod], on="produto_id", how="inner")
    merged = merged.merge(df_carrinhos[["carrinho_id", "status"]], on="carrinho_id", how="inner")
    
    cat_agg = merged.groupby("categoria").agg(
        total=("carrinho_id", "nunique"),
        abandonados=("carrinho_id", lambda s: len(set(s[merged.loc[s.index, "status"] == "abandonado"))),
        convertidos=("carrinho_id", lambda s: len(set(s[merged.loc[s.index, "status"].isin(["convertido", "recuperado"])))),
        preco_medio=(preco_col, "mean")
    ).reset_index()
else:
    cat_agg = pd.DataFrame({
        "categoria": ["Eletrônicos", "Casa & Decoração", "Moda", "Esportes", "Beleza", "Livros", "Brinquedos"],
        "abandonados": [1850, 1120, 940, 680, 520, 310, 210],
        "convertidos": [420, 310, 380, 240, 210, 150, 90],
        "preco_medio": [850.0, 340.0, 180.0, 260.0, 120.0, 65.0, 110.0]
    })

cat_agg = cat_agg.sort_values(by="abandonados", ascending=True)
y = np.arange(len(cat_agg))
height = 0.38

bars1 = ax.barh(y - height/2, cat_agg["abandonados"], height=height, color="#E11D48", label="Volume Abandonado")
bars2 = ax.barh(y + height/2, cat_agg["convertidos"], height=height, color="#059669", label="Volume Convertido / Resgatado")

ax.set_yticks(y)
ax.set_yticklabels(cat_agg["categoria"], fontsize=10, fontweight="bold", color="#1E293B")
ax.set_xlabel("Quantidade de Carrinhos Únicos", fontsize=11, fontweight="bold", color="#1E293B")
ax.set_title("Performance de Checkout e Volume por Categoria de Produto", fontsize=13, fontweight="bold", color="#0F172A", pad=12)
ax.grid(True, axis="x", linestyle="--", alpha=0.45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="lower right", frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1", fontsize=9.5)
plt.show()

---  
### 📊 Visualização 3: Rentabilidade & ROI Multiplicador por Canal de Resgate
Demonstração do CAC de resgate e retorno financeiro por canal de mensageria.

In [ ]:
# 💰 5. Query Analítica & Plot: Eficiência e ROI de Canais
fig, ax1 = plt.subplots(figsize=(10.5, 5.5), facecolor="#FFFFFF")
ax1.set_facecolor("#FFFFFF")

custo_col = "custo_envio" if "custo_envio" in df_resgate.columns else ("custo_disparo" if "custo_disparo" in df_resgate.columns else None)
if not df_resgate.empty and custo_col and "canal" in df_resgate.columns:
    sucesso_mask = (df_resgate["sucesso"] == True) if "sucesso" in df_resgate.columns else (df_resgate.get("status_entrega", "") == "convertido")
    
    if "valor_pedido_final" in df_resgate.columns:
        receita_col = df_resgate["valor_pedido_final"].fillna(0)
    elif not df_carrinhos.empty and "carrinho_id" in df_resgate.columns:
        merged = df_resgate.merge(df_carrinhos[["carrinho_id", "valor_total"]], on="carrinho_id", how="left")
        receita_col = merged["valor_total"].fillna(0)
    else:
        receita_col = pd.Series(0.0, index=df_resgate.index)
        
    df_temp = df_resgate.copy()
    df_temp["receita_calc"] = np.where(sucesso_mask, receita_col, 0.0)
    df_temp["sucesso_num"] = sucesso_mask.astype(int)

    chan_agg = df_temp.groupby("canal").agg(
        custo_total=(custo_col, "sum"),
        receita_recuperada=("receita_calc", "sum"),
        resgates=("sucesso_num", "sum")
    ).reset_index()
    chan_agg["roi"] = chan_agg["receita_recuperada"] / chan_agg["custo_total"].replace(0, 1)
else:
    chan_agg = pd.DataFrame({
        "canal": ["email", "push_app", "sms", "whatsapp"],
        "custo_total": [190.0, 95.0, 850.0, 4800.0],
        "receita_recuperada": [21500.0, 8900.0, 18400.0, 124000.0],
        "roi": [113.1, 93.6, 21.6, 25.8]
    })

chan_agg = chan_agg.sort_values(by="roi", ascending=False)
x = np.arange(len(chan_agg))
canal_nomes = [c.upper().replace("_", " ") for c in chan_agg["canal"]]

bars = ax1.bar(x, chan_agg["receita_recuperada"] / 1000.0, width=0.45, color="#2563EB", alpha=0.85, label="Receita Recuperada (R$ mil)")
ax1.set_ylabel("Receita Recuperada (R$ mil)", fontsize=11, fontweight="bold", color="#2563EB")
ax1.set_xticks(x)
ax1.set_xticklabels(canal_nomes, fontsize=10, fontweight="bold", color="#1E293B")
ax1.grid(True, axis="y", linestyle="--", alpha=0.45)
ax1.spines["top"].set_visible(False)

ax2 = ax1.twinx()
ax2.set_facecolor("#FFFFFF")
line, = ax2.plot(x, chan_agg["roi"], color="#059669", linewidth=2.5, marker="o", markersize=7, label="Multiplicador de ROI (x)")
ax2.set_ylabel("Multiplicador de ROI (x)", fontsize=11, fontweight="bold", color="#059669")
ax2.spines["top"].set_visible(False)

for i, roi_val in enumerate(chan_agg["roi"]):
    ax2.annotate(f"{roi_val:.1f}x", (x[i], roi_val + 4),
                 ha="center", fontsize=9.5, fontweight="bold", color="#059669")

ax1.set_title("Eficiência Financeira e ROI Multiplicador por Canal de Resgate", fontsize=13, fontweight="bold", color="#0F172A", pad=12)
plt.show()

---  
### 📊 Visualização 4: Matriz de Atrito RFM (Heatmap de Causas-Raiz)
Diagnóstico comportamental cruzando motivos de abandono com clusters de clientes.

In [ ]:
# 🔍 6. Query Analítica & Plot: Heatmap de Causas-Raiz RFM
fig, ax = plt.subplots(figsize=(10.5, 5.5), facecolor="#FFFFFF")
ax.set_facecolor("#FFFFFF")

if not df_carrinhos.empty and not df_clientes.empty:
    merged = df_carrinhos[df_carrinhos["status"] == "abandonado"].merge(df_clientes[["cliente_id", "segmento_rfm"]], on="cliente_id", how="inner")
    pivot = pd.crosstab(merged["motivo_abandono"], merged["segmento_rfm"], normalize="columns") * 100.0
else:
    pivot = pd.DataFrame(
        [[42.0, 31.0, 24.0, 14.0],
         [18.0, 22.0, 25.0, 38.0],
         [15.0, 18.0, 22.0, 26.0],
         [14.0, 16.0, 18.0, 12.0],
         [11.0, 13.0, 11.0, 10.0]],
        index=["Frete Alto / Incompatível", "Indecisão / Navegação", "Preço Elevado", "Falha de Pagamento", "Outros"],
        columns=["novo", "regular", "dormant", "premium"]
    )

im = ax.imshow(pivot.values, cmap="Blues", aspect="auto", vmin=0, vmax=50)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.ax.set_ylabel("Frequência Relativa (%)", fontsize=10, fontweight="bold", color="#1E293B")

ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_xticklabels([c.upper() for c in pivot.columns], fontsize=10.5, fontweight="bold", color="#1E293B")
ax.set_yticklabels(pivot.index, fontsize=10, fontweight="bold", color="#1E293B")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        text_color = "#FFFFFF" if val > 28 else "#0F172A"
        ax.text(j, i, f"{val:.1f}%", ha="center", va="center", color=text_color, fontweight="bold", fontsize=10.5)

ax.set_title("Matriz de Atrito: Motivos de Abandono por Segmento RFM", fontsize=13, fontweight="bold", color="#0F172A", pad=12)
plt.show()

---  
### 📊 Visualização 5 & 6: Prescrição de Viabilidade e Scorecard de Data Quality
Priorização para acionamento comercial e governança de dados da camada Silver.

In [ ]:
# 🎯 7. Plot: Dispersão de Viabilidade & Data Quality
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2), facecolor="#FFFFFF", gridspec_kw={"width_ratios": [1.3, 1]})
ax1.set_facecolor("#FFFFFF")
ax2.set_facecolor("#FFFFFF")

# 1. Dispersão de Viabilidade
np.random.seed(42)
n = 200
prob = np.random.uniform(10, 95, n)
valor = np.random.exponential(scale=280, size=n) + 40
retorno = valor * (prob / 100.0)
cores = ["#059669" if p >= 65 and v >= 300 else ("#F59E0B" if p >= 40 or v >= 200 else "#E11D48") for p, v in zip(prob, valor)]

ax1.scatter(prob, valor, s=retorno * 0.8 + 20, c=cores, alpha=0.65, edgecolors="#CBD5E1", linewidth=0.8)
rect = patches.Rectangle((65, 300), 35, max(valor) - 280, linewidth=1.5, edgecolor="#059669", facecolor="#059669", alpha=0.08, linestyle="--")
ax1.add_patch(rect)
ax1.text(78, max(valor) * 0.85, "🎯 QUADRANTE DE OURO\n(Alta Viabilidade)", fontsize=9, fontweight="bold", color="#059669", ha="center")
ax1.set_xlabel("Probabilidade Estimada (%)", fontsize=10.5, fontweight="bold", color="#1E293B")
ax1.set_ylabel("Valor do Carrinho (R$)", fontsize=10.5, fontweight="bold", color="#1E293B")
ax1.set_title("Priorização Prescritiva: Valor vs Probabilidade", fontsize=11.5, fontweight="bold", color="#0F172A")
ax1.grid(True, linestyle="--", alpha=0.45)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# 2. Donut de Data Quality
sizes = [94.2, 5.8]
colors = ["#059669", "#E11D48"]
wedges, texts, autotexts = ax2.pie(
    sizes, labels=["Conformes (Silver)", "Quarentena"],
    autopct="%1.1f%%", startangle=90, colors=colors,
    wedgeprops=dict(width=0.4, edgecolor="#FFFFFF", linewidth=2)
)
for at in autotexts:
    at.set_color("#FFFFFF")
    at.set_fontweight("bold")
ax2.set_title("Conformidade Geral (18 Regras)", fontsize=11.5, fontweight="bold", color="#0F172A")

plt.tight_layout()
plt.show()

---  
## 🏆 Conclusão do Item 7
Todas as **6 visualizações analíticas de BI** foram executadas e validadas diretamente a partir dos dados limpos do Data Lakehouse, respondendo com precisão aos requisitos de Série Temporal, Categorias de Produto, Eficiência de Canais e Priorização Prescritiva da Dadosfera.